In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from dataclasses import dataclass
import math
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

@dataclass
class HopfieldConfig:
    embed_dim: int = 64  # Matches ResNet18 feature output
    num_memories: int = 25
    num_classes: int = 10
    T_updates: int = 1
    proj_dim: int = 512
    entropy_reg_weight: float = 0.5
    entropy_warmup_epochs: int = 2
    blend_lower_bound: float = 2.0
    blend_upper_bound: float = 15.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    rapid_debug_mode: bool = False
    debug_batches: int = 100

cfg = HopfieldConfig()

# --- Architectures ---

class CIFARResNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        print("Downloading/Loading pre-trained CIFAR-10 ResNet20...")
        resnet = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=True)
        self.net = nn.Sequential(*list(resnet.children())[:-1], nn.Flatten())
        
        for param in self.net.parameters():
            param.requires_grad = False
            
    def forward(self, x):
        return self.net(x)
        
    def train(self, mode=True):
        super().train(mode)
        self.net.eval()
        return self

class SmallCNNEncoder(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, embed_dim)
        )
        
    def forward(self, x):
        return self.net(x)

class DiagnosticHopfield(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.memories = nn.Parameter(torch.Tensor(cfg.num_memories, cfg.embed_dim))
        self.phi = nn.Sequential(
            nn.Linear(cfg.embed_dim, cfg.embed_dim * 2),
            nn.GELU(),
            nn.Linear(cfg.embed_dim * 2, cfg.proj_dim),
            nn.LayerNorm(cfg.proj_dim)
        )
        self.beta = nn.Parameter(torch.tensor(1.0))
        self.norm = nn.LayerNorm(cfg.embed_dim)
        nn.init.normal_(self.memories, mean=0.0, std=1.0 / math.sqrt(cfg.embed_dim))

    def forward(self, state, blend_alpha=0.0):
        beta = F.softplus(self.beta)
        state_proj = self.phi(state)
        mem_proj = self.phi(self.memories)
        
        dist_sq = torch.cdist(state_proj, mem_proj, p=2.0).pow(2)
        scores = -dist_sq * beta
        
        attention = F.softmax(scores, dim=-1)
        log_attention = F.log_softmax(scores, dim=-1)
        entropy_tensor = -torch.sum(attention * log_attention, dim=-1)
        
        retrieved = torch.matmul(attention, self.memories)
        
        blended_state = (blend_alpha * state) + ((1.0 - blend_alpha) * retrieved)
        out_state = self.norm(blended_state) 
        
        return out_state, attention, entropy_tensor, retrieved

class VisionModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.hopfield = DiagnosticHopfield(cfg)
        self.classifier = nn.Linear(cfg.embed_dim, cfg.num_classes)

    def forward(self, x, blend_alpha=0.0):
        encoded = self.encoder(x)
        retrieved_norm, attention, entropy, _ = self.hopfield(encoded, blend_alpha)
        logits = self.classifier(retrieved_norm)
        return logits, entropy

# --- Training Utilities ---

def get_dataloaders():
    tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])
    ])
    train_ds = datasets.CIFAR10('./data', train=True, download=True, transform=tf)
    test_ds = datasets.CIFAR10('./data', train=False, download=True, transform=tf)
    
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
    return train_loader, test_loader

def initialize_kmeans(model, dataloader):
    print("Running MiniBatch K-Means memory initialization...")
    model.eval()
    all_embeddings = []
    
    subset_size = 2500 
    subset_indices = np.random.choice(len(dataloader.dataset), subset_size, replace=False)
    subset_loader = DataLoader(Subset(dataloader.dataset, subset_indices), batch_size=256)
    
    with torch.no_grad():
        for i, (data, _) in enumerate(subset_loader):
            if cfg.rapid_debug_mode and i >= cfg.debug_batches: break
            data = data.to(cfg.device)
            emb = model.encoder(data)
            all_embeddings.append(emb.cpu())
            
    all_embeddings = torch.cat(all_embeddings, dim=0).numpy()
    
    kmeans = MiniBatchKMeans(
        n_clusters=cfg.num_memories, 
        batch_size=1024, 
        n_init='auto', 
        random_state=42
    )
    kmeans.fit(all_embeddings)
    
    centroids = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32).to(cfg.device)
    model.hopfield.memories.data.copy_(centroids)
    
    torch.save(model.hopfield.memories.data.cpu(), "hopfield_init_memories.pt")
    print("MiniBatch K-Means initialization complete.")

def plot_pca(model, dataloader, title, filename, num_samples=1500):
    print(f"Generating PCA plot: {filename}...")
    model.eval()
    embeddings = []
    labels = []
    
    with torch.no_grad():
        for data, target in dataloader:
            data = data.to(cfg.device)
            emb = model.encoder(data)
            embeddings.append(emb.cpu())
            labels.append(target.cpu())
            
            if sum(len(b) for b in embeddings) >= num_samples:
                break
                
    embeddings = torch.cat(embeddings, dim=0)[:num_samples].numpy()
    labels = torch.cat(labels, dim=0)[:num_samples].numpy()
    memories = model.hopfield.memories.detach().cpu().numpy()
    
    pca = PCA(n_components=2)
    combined_data = np.vstack((embeddings, memories))
    pca.fit(combined_data)
    
    emb_pca = pca.transform(embeddings)
    mem_pca = pca.transform(memories)
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(emb_pca[:, 0], emb_pca[:, 1], c=labels, cmap='tab10', alpha=0.6, s=15, label='Data (Classes)')
    plt.scatter(mem_pca[:, 0], mem_pca[:, 1], c='black', marker='X', s=100, edgecolors='white', label='Memories')
    
    plt.colorbar(scatter, label='Class Label')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(filename)
    plt.close()

def evaluate(model, dataloader, criterion, blend_alpha=0.0):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for i, (data, target) in enumerate(dataloader):
            if cfg.rapid_debug_mode and i >= cfg.debug_batches: break
            data, target = data.to(cfg.device), target.to(cfg.device)
            
            logits, _ = model(data, blend_alpha=blend_alpha)
            loss = criterion(logits, target)
            
            total_loss += loss.item() * data.size(0)
            preds = logits.argmax(dim=-1)
            correct += (preds == target).sum().item()
            total += data.size(0)
            
    return total_loss / total, correct / total

def plot_curves(history, phase_name):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, ax1 = plt.subplots(figsize=(10, 6))

    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss', color='tab:red')
    ax1.plot(epochs, history['train_loss'], 'r-', label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r--', label='Val Loss')
    ax1.tick_params(axis='y', labelcolor='tab:red')
    
    ax2 = ax1.twinx()
    ax2.set_ylabel('Accuracy', color='tab:blue')
    ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    ax2.plot(epochs, history['val_acc'], 'b--', label='Val Acc')
    ax2.tick_params(axis='y', labelcolor='tab:blue')

    fig.legend(loc='center right', bbox_to_anchor=(0.95, 0.5))
    plt.title(f"{phase_name} Learning Curves")
    plt.grid(True, alpha=0.3)
    plt.savefig(f"{phase_name.lower().replace(' ', '_')}_curves.png")
    plt.close()

# --- Phase 1: Metric Space Initialization ---

def run_phase1(train_loader, test_loader):
    print(f"\n=== Starting Phase 1 {'[DEBUG MODE]' if cfg.rapid_debug_mode else ''} ===")
    model = VisionModel(CIFARResNetEncoder()).to(cfg.device)
    initialize_kmeans(model, train_loader)
    
    plot_pca(model, test_loader, "Checkpoint 1: Before Phase 1 (ResNet Encoder + KMeans)", "pca_1_before_phase1.png")
    
    base_params = [p for n, p in model.named_parameters() if 'hopfield.beta' not in n and p.requires_grad]
    optimizer = optim.AdamW([
        {'params': base_params, 'lr': 1e-3, 'weight_decay': 1e-4},
        {'params': [model.hopfield.beta], 'lr': 0.1, 'weight_decay': 0.0}
    ])
    criterion = nn.CrossEntropyLoss()
    
    epochs = 15 if not cfg.rapid_debug_mode else 3
    decay_epochs = 10 if not cfg.rapid_debug_mode else 2
    
    batches_per_epoch = cfg.debug_batches if cfg.rapid_debug_mode else len(train_loader)
    total_decay_steps = decay_epochs * batches_per_epoch
    global_step = 0
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for i, (data, target) in enumerate(train_loader):
            if cfg.rapid_debug_mode and i >= cfg.debug_batches: break
            
            data, target = data.to(cfg.device), target.to(cfg.device)
            optimizer.zero_grad()
            
            blend_alpha = max(0.0, 0.8 - (0.8 * (global_step / total_decay_steps)))
            
            logits, entropy_tensor = model(data, blend_alpha=blend_alpha)
            
            class_loss = criterion(logits, target)
            penalty_low = F.relu(math.log(cfg.blend_lower_bound) - entropy_tensor) ** 2
            penalty_high = F.relu(entropy_tensor - math.log(cfg.blend_upper_bound)) ** 2
            
            loss = class_loss + (cfg.entropy_reg_weight * (penalty_low.mean() + penalty_high.mean()))
            loss.backward()
            
            optimizer.step()
            
            epoch_loss += loss.item() * data.size(0)
            preds = logits.argmax(dim=-1)
            correct += (preds == target).sum().item()
            total += data.size(0)
            global_step += 1
            
        train_loss = epoch_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, test_loader, criterion, blend_alpha=blend_alpha)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Phase 1 - Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | End Alpha: {blend_alpha:.4f}")

    plot_curves(history, "Phase 1")
    plot_pca(model, test_loader, "Checkpoint 2: After Phase 1 (ResNet Tuned)", "pca_2_after_phase1.png")

    torch.save({
        'hopfield': model.hopfield.state_dict(),
        'classifier': model.classifier.state_dict()
    }, "phase1_weights.pt")

# --- Phase 2: Distillation and Adaptation ---

def run_phase2(train_loader, test_loader):
    print(f"\n=== Starting Phase 2 {'[DEBUG MODE]' if cfg.rapid_debug_mode else ''} ===")
    model = VisionModel(SmallCNNEncoder(cfg.embed_dim)).to(cfg.device)
    
    checkpoint = torch.load("phase1_weights.pt", weights_only=True, map_location=cfg.device)
    model.hopfield.load_state_dict(checkpoint['hopfield'])
    model.classifier.load_state_dict(checkpoint['classifier'])
    
    plot_pca(model, test_loader, "Checkpoint 3: Before Phase 2 (Untrained CNN)", "pca_3_before_phase2.png")
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    criterion = nn.CrossEntropyLoss()

    # --- Phase 2A ---
    print("-> Phase 2A: Training Encoder Only (Hopfield Frozen)")
    for param in model.hopfield.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = False
        
    optimizer_2a = optim.AdamW(model.encoder.parameters(), lr=1e-3, weight_decay=1e-4)
    epochs_2a = 10 if not cfg.rapid_debug_mode else 1
    
    for epoch in range(1, epochs_2a + 1):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for i, (data, target) in enumerate(train_loader):
            if cfg.rapid_debug_mode and i >= cfg.debug_batches: break
            
            data, target = data.to(cfg.device), target.to(cfg.device)
            optimizer_2a.zero_grad()
            logits, _ = model(data, blend_alpha=0.0)
            loss = criterion(logits, target)
            loss.backward()
            
            optimizer_2a.step()
            
            epoch_loss += loss.item() * data.size(0)
            preds = logits.argmax(dim=-1)
            correct += (preds == target).sum().item()
            total += data.size(0)
            
        train_loss = epoch_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, test_loader, criterion, blend_alpha=0.0)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Phase 2A - Epoch {epoch:02d}/{epochs_2a} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # --- Phase 2B ---
    print("-> Phase 2B: Global Unfreeze")
    for param in model.hopfield.parameters():
        param.requires_grad = True
    for param in model.classifier.parameters():
        param.requires_grad = True
        
    optimizer_2b = optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': 1e-4, 'weight_decay': 1e-4},
        {'params': model.hopfield.parameters(), 'lr': 1e-5, 'weight_decay': 1e-4},
        {'params': model.classifier.parameters(), 'lr': 1e-5, 'weight_decay': 1e-4}
    ])
    
    epochs_2b = 10 if not cfg.rapid_debug_mode else 1
    
    for epoch in range(1, epochs_2b + 1):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0
        
        for i, (data, target) in enumerate(train_loader):
            if cfg.rapid_debug_mode and i >= cfg.debug_batches: break
            
            data, target = data.to(cfg.device), target.to(cfg.device)
            optimizer_2b.zero_grad()
            logits, _ = model(data, blend_alpha=0.0)
            loss = criterion(logits, target)
            loss.backward()
            
            optimizer_2b.step()
            
            epoch_loss += loss.item() * data.size(0)
            preds = logits.argmax(dim=-1)
            correct += (preds == target).sum().item()
            total += data.size(0)
            
        train_loss = epoch_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, test_loader, criterion, blend_alpha=0.0)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Phase 2B - Epoch {epoch:02d}/{epochs_2b} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    plot_curves(history, "Phase 2")
    plot_pca(model, test_loader, "Checkpoint 4: After Phase 2 (Trained CNN)", "pca_4_after_phase2.png")
    
    torch.save(model.hopfield.memories.data.cpu(), "hopfield_final_memories.pt")

if __name__ == '__main__':
    train_loader, test_loader = get_dataloaders()
    run_phase1(train_loader, test_loader)
    run_phase2(train_loader, test_loader)


=== Starting Phase 1  ===
Downloading/Loading pre-trained CIFAR-10 ResNet20...


Using cache found in /home/aroen488/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


Running MiniBatch K-Means memory initialization...
MiniBatch K-Means initialization complete.
Generating PCA plot: pca_1_before_phase1.png...
Phase 1 - Epoch 01 | Train Loss: 0.2874 | Val Loss: 0.2893 | Train Acc: 0.9580 | Val Acc: 0.9226 | End Alpha: 0.7202
Phase 1 - Epoch 02 | Train Loss: 0.0122 | Val Loss: 0.3411 | Train Acc: 0.9989 | Val Acc: 0.9230 | End Alpha: 0.6402
Phase 1 - Epoch 03 | Train Loss: 0.0059 | Val Loss: 0.3630 | Train Acc: 0.9991 | Val Acc: 0.9232 | End Alpha: 0.5602
Phase 1 - Epoch 04 | Train Loss: 0.0041 | Val Loss: 0.4214 | Train Acc: 0.9991 | Val Acc: 0.9239 | End Alpha: 0.4802
Phase 1 - Epoch 05 | Train Loss: 0.0031 | Val Loss: 0.4651 | Train Acc: 0.9992 | Val Acc: 0.9230 | End Alpha: 0.4002
Phase 1 - Epoch 06 | Train Loss: 0.0030 | Val Loss: 0.4989 | Train Acc: 0.9992 | Val Acc: 0.9216 | End Alpha: 0.3202
Phase 1 - Epoch 07 | Train Loss: 0.0031 | Val Loss: 0.4864 | Train Acc: 0.9991 | Val Acc: 0.9219 | End Alpha: 0.2402
Phase 1 - Epoch 08 | Train Loss: 0.0033